# Prerequirements & Setup
Start by making sure you have the required files in the proper structure before trying this, because it requires you to mount the drive in order to access the python scripts and datasets. The files should be accessible via our repo.

## Installing Dependencies
First, we need to install the required Python packages that are not pre-installed in Google Colab. This includes `transformers` for the language models, `datasets` for loading data, and `openai` (if your scripts interact with OpenAI's API).

In [ ]:
# Install necessary dependencies not pre-installed in Colab
!pip install transformers datasets openai

In [ ]:
# Consolidated imports from run.py, custom_datasets.py, and calculate_metrics.py
import argparse
import datetime
import functools
import json
import os
import random
import re
import time
from multiprocessing.pool import ThreadPool

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import tqdm
import transformers

from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve
)

In [ ]:
# Check if PyTorch is using the GPU version and if a GPU is available
print("PyTorch version:", torch.__version__)
print("Is CUDA (GPU) available?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

## Mounting Google Drive
Since the project files (like `run.py`) and datasets are stored in your Google Drive, we need to mount it to the Colab environment. When you run this, Colab will prompt you to grant access to your Drive.

**Note:** In the code cell after this one, you MUST update the `PROJECT_PATH` variable to point to the exact folder where you uploaded the DetectGPT files.

In [ ]:
# from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Define the path to your project folder. Where ever you set your project folder to be, set it here
# Example:
# PROJECT_PATH = "/content/drive/MyDrive/{YOUR FOLDER LOCATION}"
PROJECT_PATH = "/content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT"


# Change the current working directory to your project folder
os.chdir(PROJECT_PATH)
print("Current Working Directory:", os.getcwd())

# Set up a cache directory in your drive for HuggingFace models
cache_dir = os.path.join(PROJECT_PATH, "hf_cache")
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir

print("HuggingFace cache set to:", os.environ['HF_HOME'])

# Running DetectGPT



## Run Flags Reference
| Flag | Required | Example | Meaning |
|---|---|---|---|
| --output_name | Recommended | smoke_hc3 | Name of the run output folder so results are easy to identify. |
| --dataset | Yes | hc3_all | Dataset key to load and evaluate. |
| --base_model_name | Yes | gpt2-medium | Base causal language model used for generation and scoring. |
| --mask_filling_model_name | Yes (for perturbation runs) | t5-base | Mask infilling model used to create perturbed text. |
| --n_perturbation_list | Yes (for perturbation runs) | 3 or 1,3,10 | Number of perturbations per sample. You can pass a comma-separated list. |
| --n_samples | Yes | 50 | Number of examples to evaluate. Smaller is faster. |
| --pct_words_masked | Recommended | 0.3 | Approximate fraction of words to mask before refill perturbation. |
| --skip_baselines | Optional switch | include flag only | If present, skips baseline metrics and runs perturbation experiments only. |
| --cache_dir | Recommended | hf_cache (if running in colab) or C:/hf_home (if running on local) | Directory for model/dataset cache to avoid repeated downloads. |

## Other Useful optional flags:
| Flag | Needed | Example value | What it means |
|---|---|---|---|
| --batch_size | Optional | 10 | Batch size for scoring/supervised loops. Lower if memory is tight. |
| --chunk_size | Optional | 10 | Chunk size for perturbation generation. Lower if VRAM/RAM is limited. |
| --span_length | Optional | 2 | Number of contiguous tokens per masked span. |
| --n_perturbation_rounds | Optional | 1 | How many times perturbation is recursively applied. |
--prompt_tokens | Optional | 64 | Number of tokens from the original prompt that is used as context by the base model for the generation of new text. The more tokens, the more context the base model will have.

## Small Smoke Test Command
```shell
python run.py --output_name smoke_hc3 --dataset hc3_all --base_model_name gpt2 --mask_filling_model_name t5-small --n_perturbation_list 3 --n_samples 50 --pct_words_masked 0.3 --skip_baselines --cache_dir C:/hf_home --batch_size 10 --chunk_size 10
```
## Reusable Template
```shell
python run.py --output_name YOUR_RUN_NAME --dataset YOUR_DATASET --base_model_name gpt2-medium --mask_filling_model_name t5-base --n_perturbation_list 3 --n_samples 500 --pct_words_masked 0.3 --skip_baselines --cache_dir hf_cache
```





## Calculating Metrics
After generating the perturbations and scoring them in the steps above, we need to calculate the final evaluation metrics (like ROC AUC).

To calculate the metrics, you will need the outputted json file from running DetectGPT. They will usually look like this:

```perturbation_3_z_results.json```

⚠️ **IMPORTANT FOR NEW USERS:** The `--path` argument in the cell below currently points to a specific result file. **You must update this path** to match the actual `.json` file generated in your `results` folder from the runs you just completed above. This command assumes that the results folder is in the project working directory.

In [ ]:
# This is an example. The path will be unique most of the time, because it includes a time stamp.
# The console log will usually say where it was saved to.
# !python calculate_metrics.py --path results/smoke_hc3/gpt2-t5-small-temp/2026-04-13-06-58-35-361222-fp32-0.3-1-hc3_all-50/perturbation_3_z_results.json

# Smoke Test
This is to make sure that everything is working.

## Tiny 50: Using only 50 samples and 5 perturbations.

In [ ]:
!python run.py --output_name tiny_50 --dataset hc3_all_10000 --base_model_name gpt2 --mask_filling_model_name t5-small --n_perturbation_list 5 --n_samples 50 --skip_baselines --cache_dir hf_cache

In [ ]:
# Replace {PUT PATH HERE} with the actual path to the results.json.
!python calculate_metrics.py --path results/tiny_50/{PUT PATH HERE}/perturbation_5_z_results.json

# Actual Tests

It can take more than 2 hours to finish running each test, due to the large number of perturbations (20 currently) for each sample, as well as calculating the logistic probability of the tokens.

## HC3 + GPT2-Large + 10000 Samples

In [ ]:
!python run.py --output_name Final_GPT2 --dataset hc3_all_10000 --base_model_name openai-community/gpt2-large --mask_filling_model_name t5-large --n_perturbation_list 20 --n_samples 10000 --cache_dir hf_cache --skip_baselines

In [ ]:
!python calculate_metrics.py --path results/Final_GPT2/{REPLACE WITH PATH HERE}/perturbation_20_z_results.json

## HC3 + Falcon + 10000 Samples

In [ ]:
!python run.py --output_name Final_Falcon --dataset hc3_all_10000 --base_model_name tiiuae/falcon-7b --mask_filling_model_name t5-large --n_perturbation_list 20 --n_samples 10000 --cache_dir hf_cache --skip_baselines

In [ ]:
!python calculate_metrics.py --path results/Final_Falcon/{REPLACE WITH PATH HERE}/perturbation_20_z_results.json

## HC3 + Falcon Instruct + 10000 Samples

In [ ]:
!python run.py --output_name Final_FalInstruct --dataset hc3_all_10000 --base_model_name tiiuae/falcon-7b-instruct --mask_filling_model_name t5-large --n_perturbation_list 20 --n_samples 10000 --cache_dir hf_cache --skip_baselines

In [ ]:
!python calculate_metrics.py --path results/Final_FalInstruct/{REPLACE WITH PATH HERE}/perturbation_20_z_results.json

# Avoidance Techniques

## Recursive Paraphrasing

In [ ]:
!python avoidance_run.py --cache_dir hf_cache --skip_baselines --dataset avoidance_recursive_hc3 --n_samples 5000 --n_perturbation_list 20 --prompt_tokens 64 --mask_filling_model_name t5-large

In [ ]:
!python calculate_metrics.py --path results/gpt2-medium-t5-large-temp/2026-04-22-22-32-16-944177-fp32-0.3-1-avoidance_recursive_hc3-5000/perturbation_20_d_results.json

## Stylistic Cleanup

In [ ]:
!python avoidance_run.py --cache_dir hf_cache --skip_baselines --dataset hc3_stylisticCleanup --n_samples 5000 --n_perturbation_list 20 --prompt_tokens 64 --pct_words_masked 0.2 --mask_filling_model_name t5-large

In [ ]:
!python calculate_metrics.py --path results/gpt2-medium-t5-large-temp/2026-04-23-06-19-45-042224-fp32-0.2-1-hc3_stylisticCleanup-5000/perturbation_20_d_results.json

## Perturbed Text

In [ ]:
!python avoidance_run.py --cache_dir hf_cache --skip_baselines --dataset hc3_perturbed --n_samples 5000 --n_perturbation_list 20 --prompt_tokens 64 --pct_words_masked 0.2 --mask_filling_model_name t5-large

In [ ]:
!python calculate_metrics.py --path results/gpt2-medium-t5-large-temp/2026-04-23-17-00-01-128245-fp32-0.2-1-hc3_perturbed-5000/perturbation_20_d_results.json